# 02 — Feature Engineering

**Goal:** turn the raw `application_train` + `bureau` + `bureau_balance` tables into a single applicant-level matrix ready for LightGBM.

Steps:
1. Clean `DAYS_EMPLOYED` sentinel.
2. Application-level engineered features (ratios, EXT_SOURCE aggregates).
3. Aggregate `bureau_balance` → one row per `SK_ID_BUREAU`.
4. Aggregate `bureau` (+ the bureau_balance summary) → one row per `SK_ID_CURR`.
5. Merge everything onto `application` and persist to `outputs/features.parquet`.

Categoricals are converted to `category` dtype so LightGBM can consume them natively (no one-hot).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import OUTPUTS_DIR, fix_days_employed, load_application, load_bureau

OUTPUTS_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 50)

## 1. Load + clean application

In [2]:
app = load_application("train")
app = fix_days_employed(app)
print(f"application shape: {app.shape}")
print(f"DAYS_EMPLOYED_ANOM positives: {app['DAYS_EMPLOYED_ANOM'].sum():,}")

application shape: (307511, 123)
DAYS_EMPLOYED_ANOM positives: 55,374


## 2. Application-level engineered features

Ratios that domain experts care about (debt-to-income, loan-to-income, employment share of life) plus EXT_SOURCE aggregates. None of these leak the target.

In [3]:
def add_application_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Income / credit ratios
    df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["CREDIT_ANNUITY_RATIO"] = df["AMT_CREDIT"] / df["AMT_ANNUITY"]  # ~loan term in years
    df["CREDIT_GOODS_RATIO"] = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"]
    df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"].clip(lower=1)
    # Life-stage ratios (DAYS_* are negative = days before application)
    df["EMPLOYED_AGE_RATIO"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
    df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365.25).round(1)
    df["EMPLOYED_YEARS"] = -df["DAYS_EMPLOYED"] / 365.25
    # EXT_SOURCE aggregates — the strongest signal in the dataset
    ext = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    df["EXT_SOURCE_MEAN"] = ext.mean(axis=1)
    df["EXT_SOURCE_MIN"] = ext.min(axis=1)
    df["EXT_SOURCE_MAX"] = ext.max(axis=1)
    df["EXT_SOURCE_STD"] = ext.std(axis=1)
    df["EXT_SOURCE_PROD"] = ext.fillna(1).prod(axis=1)  # zero if any is exactly 0
    df["EXT_SOURCE_NAN_COUNT"] = ext.isna().sum(axis=1)
    return df

app = add_application_features(app)
new_cols = [
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_ANNUITY_RATIO",
    "CREDIT_GOODS_RATIO", "INCOME_PER_PERSON", "EMPLOYED_AGE_RATIO",
    "AGE_YEARS", "EMPLOYED_YEARS",
    "EXT_SOURCE_MEAN", "EXT_SOURCE_MIN", "EXT_SOURCE_MAX", "EXT_SOURCE_STD",
    "EXT_SOURCE_PROD", "EXT_SOURCE_NAN_COUNT",
]
print(app[new_cols].describe().T[["mean", "std", "min", "max"]])

                              mean            std           min           max
CREDIT_INCOME_RATIO       3.957570       2.689728  4.807615e-03  8.473684e+01
ANNUITY_INCOME_RATIO      0.180930       0.094574  2.238846e-04  1.875965e+00
CREDIT_ANNUITY_RATIO     21.612322       7.823823  8.036674e+00  4.530508e+01
CREDIT_GOODS_RATIO        1.122995       0.124045  1.500000e-01  6.000000e+00
INCOME_PER_PERSON     93105.879645  101373.363394  2.812500e+03  3.900000e+07
EMPLOYED_AGE_RATIO        0.156861       0.133549 -0.000000e+00  7.288115e-01
AGE_YEARS                43.906968      11.947987  2.050000e+01  6.910000e+01
EMPLOYED_YEARS            6.527500       6.402081 -0.000000e+00  4.904038e+01
EXT_SOURCE_MEAN           0.509251       0.149802  5.939651e-06  8.789034e-01
EXT_SOURCE_MIN            0.399582       0.187425  8.173617e-08  8.789034e-01
EXT_SOURCE_MAX            0.615882       0.156130  5.939651e-06  9.626928e-01
EXT_SOURCE_STD            0.151246       0.099911  3.538459e-07 

## 3. Aggregate `bureau_balance` → per `SK_ID_BUREAU`

`bureau_balance` has one row per (bureau record, month) with a status code (`C`=closed, `X`=unknown, `0`–`5`=days-past-due buckets). We collapse to per-bureau summaries.

In [4]:
bureau, bureau_balance = load_bureau()
print(f"bureau: {bureau.shape}, bureau_balance: {bureau_balance.shape}")

# Encode STATUS: C/X stay categorical, 0-5 → numeric DPD bucket
bb = bureau_balance.copy()
bb["STATUS_DPD"] = pd.to_numeric(bb["STATUS"], errors="coerce")  # NaN for C/X
bb["STATUS_IS_DPD"] = (bb["STATUS_DPD"] > 0).astype("int8")
bb["STATUS_IS_CLOSED"] = (bb["STATUS"] == "C").astype("int8")

bb_agg = bb.groupby("SK_ID_BUREAU").agg(
    BB_MONTHS_COUNT=("MONTHS_BALANCE", "size"),
    BB_MONTHS_MIN=("MONTHS_BALANCE", "min"),
    BB_STATUS_DPD_MAX=("STATUS_DPD", "max"),
    BB_STATUS_DPD_MEAN=("STATUS_DPD", "mean"),
    BB_DPD_MONTHS=("STATUS_IS_DPD", "sum"),
    BB_CLOSED_MONTHS=("STATUS_IS_CLOSED", "sum"),
).reset_index()
print(f"bureau_balance → {bb_agg.shape[0]:,} bureau records summarized")
bb_agg.head()

bureau: (1716428, 17), bureau_balance: (27299925, 3)


bureau_balance → 817,395 bureau records summarized


,SK_ID_BUREAU,BB_MONTHS_COUNT,BB_MONTHS_MIN,BB_STATUS_DPD_MAX,BB_STATUS_DPD_MEAN,BB_DPD_MONTHS,BB_CLOSED_MONTHS
0,5001709,97,-96,NaN,NaN,0,86
1,5001710,83,-82,0.0,0.0,0,48
2,5001711,4,-3,0.0,0.0,0,0
3,5001712,19,-18,0.0,0.0,0,9
4,5001713,22,-21,NaN,NaN,0,0


## 4. Aggregate `bureau` (+ bb summary) → per `SK_ID_CURR`

For each applicant we want counts, sums, means, and maxes of their prior bureau records. We compute these once over the full table and then *separately* over active credits (the riskier ones).

In [5]:
bureau_full = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")

NUMERIC_AGGS = {
    "DAYS_CREDIT": ["min", "max", "mean"],
    "DAYS_CREDIT_ENDDATE": ["min", "max", "mean"],
    "DAYS_CREDIT_UPDATE": ["mean"],
    "CREDIT_DAY_OVERDUE": ["max", "mean"],
    "AMT_CREDIT_MAX_OVERDUE": ["max", "mean"],
    "AMT_CREDIT_SUM": ["sum", "mean", "max"],
    "AMT_CREDIT_SUM_DEBT": ["sum", "mean", "max"],
    "AMT_CREDIT_SUM_LIMIT": ["sum", "mean"],
    "AMT_CREDIT_SUM_OVERDUE": ["sum", "mean", "max"],
    "AMT_ANNUITY": ["sum", "mean", "max"],
    "CNT_CREDIT_PROLONG": ["sum", "max"],
    "BB_MONTHS_COUNT": ["sum", "mean"],
    "BB_STATUS_DPD_MAX": ["max", "mean"],
    "BB_DPD_MONTHS": ["sum", "max"],
}

def aggregate(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    agg = df.groupby("SK_ID_CURR").agg(NUMERIC_AGGS)
    agg.columns = [f"{prefix}_{col}_{stat.upper()}" for col, stat in agg.columns]
    agg[f"{prefix}_COUNT"] = df.groupby("SK_ID_CURR").size()
    return agg.reset_index()

bureau_all = aggregate(bureau_full, "BURO")
bureau_active = aggregate(bureau_full[bureau_full["CREDIT_ACTIVE"] == "Active"], "BURO_ACT")
bureau_closed = aggregate(bureau_full[bureau_full["CREDIT_ACTIVE"] == "Closed"], "BURO_CLO")

print(f"All-credit aggregates:    {bureau_all.shape}")
print(f"Active-credit aggregates: {bureau_active.shape}")
print(f"Closed-credit aggregates: {bureau_closed.shape}")

All-credit aggregates:    (305811, 35)
Active-credit aggregates: (251815, 35)
Closed-credit aggregates: (267925, 35)


## 5. Merge bureau aggregates onto application

In [6]:
features = (
    app
    .merge(bureau_all, on="SK_ID_CURR", how="left")
    .merge(bureau_active, on="SK_ID_CURR", how="left")
    .merge(bureau_closed, on="SK_ID_CURR", how="left")
)

# Ratio of active vs total bureau debt — captures current leverage
features["BURO_ACTIVE_DEBT_RATIO"] = (
    features["BURO_ACT_AMT_CREDIT_SUM_DEBT_SUM"]
    / features["BURO_AMT_CREDIT_SUM_DEBT_SUM"].replace(0, np.nan)
)
features["BURO_ACTIVE_COUNT_RATIO"] = (
    features["BURO_ACT_COUNT"] / features["BURO_COUNT"]
)

print(f"Final shape: {features.shape}")
print(f"Columns added vs raw application: {features.shape[1] - app.shape[1]}")

Final shape: (307511, 241)
Columns added vs raw application: 104


## 6. Cast categoricals + persist

LightGBM consumes pandas `category` dtype directly. Saving as parquet preserves dtypes and is ~10x faster to reload than CSV.

In [7]:
cat_cols = features.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    features[col] = features[col].astype("category")
print(f"Cast {len(cat_cols)} columns to category dtype")

out_path = OUTPUTS_DIR / "features.parquet"
features.to_parquet(out_path, index=False)
print(f"Wrote {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")
print(f"Shape on disk: {features.shape}")

Cast 16 columns to category dtype


Wrote C:\Users\khera\finance-ml-cybersec project\outputs\features.parquet (86.3 MB)
Shape on disk: (307511, 241)


## Sanity checks before modeling

In [8]:
# Target preserved, no row loss, no accidental NaN in target
assert features["TARGET"].notna().all(), "TARGET has NaN — merge introduced row loss"
assert features["SK_ID_CURR"].is_unique, "SK_ID_CURR not unique — bureau aggregates didn't collapse properly"
assert len(features) == len(app), f"Row count changed: {len(app)} → {len(features)}"

print(f"Default rate preserved: {features['TARGET'].mean():.4f}")
print(f"Applicants with any bureau history: {features['BURO_COUNT'].notna().sum():,} ({features['BURO_COUNT'].notna().mean()*100:.1f}%)")
print("\nTop new bureau features by correlation magnitude with TARGET:")
buro_cols = [c for c in features.columns if c.startswith("BURO")]
corrs = features[buro_cols + ["TARGET"]].corr()["TARGET"].drop("TARGET").dropna()
print(corrs.reindex(corrs.abs().sort_values(ascending=False).index).head(15))

Default rate preserved: 0.0807
Applicants with any bureau history: 263,491 (85.7%)

Top new bureau features by correlation magnitude with TARGET:


BURO_DAYS_CREDIT_MEAN              0.089729
BURO_BB_MONTHS_COUNT_MEAN         -0.080193
BURO_ACTIVE_COUNT_RATIO            0.075393
BURO_DAYS_CREDIT_MIN               0.075248
BURO_DAYS_CREDIT_UPDATE_MEAN       0.068927
BURO_ACT_BB_MONTHS_COUNT_MEAN     -0.065154
BURO_ACT_DAYS_CREDIT_MEAN          0.064040
BURO_CLO_DAYS_CREDIT_MIN           0.061191
BURO_ACT_COUNT                     0.060928
BURO_ACT_DAYS_CREDIT_MAX           0.060410
BURO_CLO_DAYS_CREDIT_MEAN          0.058488
BURO_CLO_BB_MONTHS_COUNT_MEAN     -0.058354
BURO_DAYS_CREDIT_MAX               0.049782
BURO_DAYS_CREDIT_ENDDATE_MEAN      0.046983
BURO_CLO_BB_STATUS_DPD_MAX_MEAN    0.044643
Name: TARGET, dtype: float64
